# 🛣️ Klasifikasi Kerusakan Jalan menggunakan ResNet-50
**Transfer Learning | TensorFlow/Keras | Google Colab**

---
### Kelas Klasifikasi:
- ✅ **Baik** — Jalan dalam kondisi normal, tidak ada kerusakan
- 🟡 **Rusak Ringan** — Kerusakan kecil, retak halus atau tambalan kecil
- 🟠 **Rusak Sedang** — Kerusakan cukup signifikan, lubang kecil atau retak lebar
- 🔴 **Rusak Berat** — Kerusakan parah, lubang besar atau permukaan hancur

### Pembagian Dataset: 70% Train / 15% Val / 15% Test

---
## 📦 STEP 1: Import Library

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os, warnings
warnings.filterwarnings('ignore')

print('✅ TensorFlow version :', tf.__version__)
print('✅ GPU tersedia       :', tf.config.list_physical_devices('GPU'))

---
## 📁 STEP 2: Konfigurasi Path & Parameter

In [ ]:
# ============================================================
# SESUAIKAN PATH INI DENGAN LOKASI DATASET KAMU
# ============================================================
TRAIN_DIR = '/content/dataset/train'
VAL_DIR   = '/content/dataset/val'
TEST_DIR  = '/content/dataset/test'

IMG_SIZE    = (224, 224)  # ResNet-50 butuh 224x224
BATCH_SIZE  = 32
EPOCHS      = 30
NUM_CLASSES = 4

# Nama folder harus sama persis dengan folder di dataset
CLASS_NAMES  = ['baik', 'rusak_berat', 'rusak_ringan', 'rusak_sedang']
CLASS_LABELS = ['Baik', 'Rusak Berat', 'Rusak Ringan', 'Rusak Sedang']

print('✅ Konfigurasi selesai')
print(f'   Kelas : {CLASS_NAMES}')

---
## 📂 STEP 3: Upload Dataset

Struktur folder yang diharapkan:
```
dataset/
├── train/
│   ├── baik/
│   ├── rusak_ringan/
│   ├── rusak_sedang/
│   └── rusak_berat/
├── val/
│   ├── baik/
│   ├── rusak_ringan/
│   ├── rusak_sedang/
│   └── rusak_berat/
└── test/
    ├── baik/
    ├── rusak_ringan/
    ├── rusak_sedang/
    └── rusak_berat/
```

In [ ]:
# OPSI 1: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/dataset_jalan' '/content/dataset'

# OPSI 2: Upload ZIP dari lokal
# from google.colab import files
# import zipfile
# uploaded = files.upload()
# with zipfile.ZipFile('dataset_jalan.zip', 'r') as z:
#     z.extractall('/content/dataset')

# Cek distribusi dataset
print('📊 Distribusi Dataset:')
print('-' * 45)
total_all = 0
for split in ['train', 'val', 'test']:
    path = f'/content/dataset/{split}'
    if os.path.exists(path):
        print(f'\n📁 {split.upper()}:')
        split_total = 0
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                split_total += count
                print(f'   {cls:15s}: {count} gambar')
        print(f'   {"TOTAL":15s}: {split_total} gambar')
        total_all += split_total
    else:
        print(f'❌ {split} folder tidak ditemukan!')
print(f'\n✅ TOTAL KESELURUHAN: {total_all} gambar')

---
## 🔄 STEP 4: Preprocessing & Augmentasi Data

In [ ]:
# TRAIN → Augmentasi + Preprocessing ResNet-50
# Augmentasi mensimulasikan kondisi foto jalan yang berbeda-beda:
# sudut kamera berbeda, pencahayaan pagi/siang/mendung, dll.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],  # Variasi pencahayaan
    fill_mode='nearest'
)

# VAL & TEST → Hanya preprocessing, TANPA augmentasi
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print('\n📊 Ringkasan Dataset:')
print(f'   Train : {train_generator.samples} gambar')
print(f'   Val   : {val_generator.samples} gambar')
print(f'   Test  : {test_generator.samples} gambar')
print(f'   Kelas : {train_generator.class_indices}')

---
## 🖼️ STEP 5: Visualisasi Sample & Distribusi Data

In [ ]:
# --- 5A: Sample Gambar ---
viz_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=8,
    class_mode='categorical', shuffle=True
)
sample_imgs, sample_lbls = next(viz_gen)
cls_list = list(viz_gen.class_indices.keys())

plt.figure(figsize=(14, 6))
for i in range(8):
    plt.subplot(2, 4, i+1)
    plt.imshow(sample_imgs[i])
    label = cls_list[np.argmax(sample_lbls[i])].replace('_', ' ').title()
    plt.title(label, fontsize=10, fontweight='bold')
    plt.axis('off')
plt.suptitle('Sample Dataset — Klasifikasi Kerusakan Jalan', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_dataset_jalan.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 5B: Distribusi Kelas Training ---
class_counts = {}
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        class_counts[cls.replace('_', ' ').title()] = len(os.listdir(cls_path))

colors = ['#4CAF50', '#F44336', '#FF9800', '#FF5722']
plt.figure(figsize=(8, 5))
bars = plt.bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(int(bar.get_height())), ha='center', fontsize=11, fontweight='bold')
plt.title('Distribusi Kelas — Data Training', fontsize=13, fontweight='bold')
plt.xlabel('Kelas Kerusakan Jalan')
plt.ylabel('Jumlah Gambar')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/distribusi_kelas_jalan.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🏗️ STEP 6: Bangun Model ResNet-50 (Transfer Learning)

In [ ]:
def build_resnet50_road(input_shape=(224, 224, 3), num_classes=4):
    """
    ResNet-50 Transfer Learning — Klasifikasi Kerusakan Jalan
    Base  : ResNet-50 pretrained ImageNet (frozen di Phase 1)
    Head  : Custom classifier 4 kelas (Baik, Rusak Ringan, Sedang, Berat)
    """
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False  # Frozen di Phase 1

    inputs  = keras.Input(shape=input_shape, name='input_jalan')
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D(name='gap')(x)
    x       = layers.Dense(512, activation='relu', name='fc_512')(x)
    x       = layers.BatchNormalization(name='bn_1')(x)
    x       = layers.Dropout(0.5, name='dropout_1')(x)
    x       = layers.Dense(256, activation='relu', name='fc_256')(x)
    x       = layers.BatchNormalization(name='bn_2')(x)
    x       = layers.Dropout(0.3, name='dropout_2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output_kerusakan')(x)

    model = keras.Model(inputs, outputs, name='ResNet50_Klasifikasi_Jalan')
    return model, base_model

resnet_model, resnet_base = build_resnet50_road()
resnet_model.summary()

trainable = sum([tf.size(w).numpy() for w in resnet_model.trainable_weights])
frozen    = sum([tf.size(w).numpy() for w in resnet_model.non_trainable_weights])
print(f'\n📊 Informasi Model:')
print(f'   Total Parameter   : {resnet_model.count_params():,}')
print(f'   Trainable (aktif) : {trainable:,}')
print(f'   Frozen (dikunci)  : {frozen:,}')

---
## ⚙️ STEP 7: Compile Model

In [ ]:
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('✅ Model berhasil dikompilasi')
print('   Optimizer : Adam (lr=0.001)')
print('   Loss      : Categorical Crossentropy')
print('   Metrics   : Accuracy')

---
## 🎯 STEP 8: Callbacks

In [ ]:
callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/resnet50_jalan_phase1_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]
print('✅ Callbacks Phase 1 siap')

---
## 🚀 STEP 9: Training Phase 1 — Frozen Base

In [ ]:
print('=' * 60)
print('🚀 PHASE 1: Training dengan Base ResNet-50 Frozen')
print('   Hanya custom classifier head yang dilatih')
print('=' * 60)

history_phase1 = resnet_model.fit(
    train_generator, epochs=15,
    validation_data=val_generator,
    callbacks=callbacks_phase1, verbose=1
)

best_p1 = max(history_phase1.history['val_accuracy'])
print(f'\n✅ Phase 1 Selesai! Best Val Accuracy: {best_p1*100:.2f}%')

---
## 🔓 STEP 10: Training Phase 2 — Fine-tuning

In [ ]:
# Unfreeze 30 layer terakhir
resnet_base.trainable = True
for layer in resnet_base.layers[:-30]:
    layer.trainable = False

resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/resnet50_jalan_phase2_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-8, verbose=1)
]

print('=' * 60)
print('🔓 PHASE 2: Fine-tuning (30 layer terakhir di-unfreeze)')
print('   Learning Rate diturunkan → 1e-5')
print('=' * 60)

history_phase2 = resnet_model.fit(
    train_generator, epochs=15,
    validation_data=val_generator,
    callbacks=callbacks_phase2, verbose=1
)

best_p2 = max(history_phase2.history['val_accuracy'])
print(f'\n✅ Phase 2 Selesai! Best Val Accuracy: {best_p2*100:.2f}%')

---
## 📊 STEP 11: Visualisasi Training History

In [ ]:
def plot_history(history, title, save_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    ax1.plot(history.history['accuracy'],     label='Train', color='#2196F3', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val',   color='#FF5722', linewidth=2, linestyle='--')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_ylim(0, 1.05)
    ax2.plot(history.history['loss'],     label='Train', color='#2196F3', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val',   color='#FF5722', linewidth=2, linestyle='--')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/{save_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Tersimpan: {save_name}.png')

plot_history(history_phase1, 'ResNet-50 Phase 1 — Kerusakan Jalan (Frozen)', 'resnet50_jalan_history_p1')
plot_history(history_phase2, 'ResNet-50 Phase 2 — Kerusakan Jalan (Fine-tuning)', 'resnet50_jalan_history_p2')

---
## 📈 STEP 12: Evaluasi pada Test Set

In [ ]:
print('=' * 60)
print('📈 Evaluasi ResNet-50 pada Test Set')
print('=' * 60)
test_loss, test_acc = resnet_model.evaluate(test_generator, verbose=1)
print(f'\n🏆 Test Accuracy : {test_acc*100:.2f}%')
print(f'   Test Loss     : {test_loss:.4f}')

---
## 🔢 STEP 13: Confusion Matrix

In [ ]:
test_generator.reset()
y_pred_prob      = resnet_model.predict(test_generator, verbose=1)
y_pred           = np.argmax(y_pred_prob, axis=1)
y_true           = test_generator.classes
class_names_list = list(test_generator.class_indices.keys())
class_labels     = [c.replace('_', ' ').title() for c in class_names_list]

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.5, annot_kws={'size': 12})
plt.title('Confusion Matrix — ResNet-50\nKlasifikasi Kerusakan Jalan', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Aktual', fontsize=12)
plt.xlabel('Prediksi', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('/content/resnet50_jalan_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix tersimpan!')

---
## 📋 STEP 14: Classification Report

In [ ]:
print('=' * 60)
print('📋 Classification Report — ResNet-50 Klasifikasi Jalan')
print('=' * 60)
report = classification_report(y_true, y_pred, target_names=class_labels)
print(report)

with open('/content/resnet50_jalan_classification_report.txt', 'w') as f:
    f.write('Classification Report\n')
    f.write('Model  : ResNet-50 (Transfer Learning)\n')
    f.write('Task   : Klasifikasi Kerusakan Jalan\n')
    f.write('Kelas  : Baik | Rusak Ringan | Rusak Sedang | Rusak Berat\n')
    f.write('='*60 + '\n')
    f.write(report)
    f.write(f'\nTest Accuracy : {test_acc*100:.2f}%')
    f.write(f'\nTest Loss     : {test_loss:.4f}')
print('\n✅ Report tersimpan!')

---
## 📊 STEP 15: Grafik Precision, Recall & F1-Score Per Kelas

In [ ]:
report_dict = classification_report(y_true, y_pred, target_names=class_labels, output_dict=True)
precision = [report_dict[c]['precision'] for c in class_labels]
recall    = [report_dict[c]['recall']    for c in class_labels]
f1        = [report_dict[c]['f1-score']  for c in class_labels]

x, width = np.arange(len(class_labels)), 0.25
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - width, precision, width, label='Precision', color='#2196F3', edgecolor='black')
b2 = ax.bar(x,         recall,    width, label='Recall',    color='#4CAF50', edgecolor='black')
b3 = ax.bar(x + width, f1,        width, label='F1-Score',  color='#FF5722', edgecolor='black')
for bars in [b1, b2, b3]:
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Kelas Kerusakan Jalan', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision, Recall & F1-Score per Kelas\nResNet-50 — Klasifikasi Kerusakan Jalan', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(class_labels)
ax.set_ylim(0, 1.2); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/resnet50_jalan_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik per-kelas tersimpan!')

---
## 💾 STEP 16: Simpan Model & Ringkasan Akhir

In [ ]:
resnet_model.save('/content/resnet50_jalan_final.h5')
print('✅ Model tersimpan: resnet50_jalan_final.h5')
print('\n' + '='*60)
print('🏆 RINGKASAN HASIL — ResNet-50 Klasifikasi Kerusakan Jalan')
print('='*60)
print(f'   Best Val Accuracy Phase 1 : {max(history_phase1.history["val_accuracy"])*100:.2f}%')
print(f'   Best Val Accuracy Phase 2 : {max(history_phase2.history["val_accuracy"])*100:.2f}%')
print(f'   Test Accuracy (Final)      : {test_acc*100:.2f}%')
print(f'   Test Loss     (Final)      : {test_loss:.4f}')
print('='*60)

# Download (opsional)
# from google.colab import files
# files.download('/content/resnet50_jalan_final.h5')

---
## 🔍 STEP 17: Prediksi Gambar Jalan Baru

In [ ]:
from tensorflow.keras.preprocessing import image

COLOR_MAP = {
    'Baik'         : 'green',
    'Rusak Ringan' : 'orange',
    'Rusak Sedang' : 'darkorange',
    'Rusak Berat'  : 'red'
}

def predict_road_image(img_path):
    img       = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = preprocess_input(image.img_to_array(img))
    img_array = np.expand_dims(img_array, axis=0)

    preds      = resnet_model.predict(img_array, verbose=0)[0]
    pred_class = class_labels[np.argmax(preds)]
    confidence = np.max(preds) * 100
    color      = COLOR_MAP.get(pred_class, 'black')

    plt.figure(figsize=(5, 5))
    plt.imshow(image.load_img(img_path))
    plt.title(f'Kondisi Jalan : {pred_class}\nKepercayaan  : {confidence:.1f}%',
              fontsize=12, fontweight='bold', color=color)
    plt.axis('off'); plt.tight_layout(); plt.show()

    print('\n📊 Probabilitas per kelas:')
    for cls, prob in zip(class_labels, preds):
        bar = '█' * int(prob * 30)
        print(f'   {cls:15s}: {prob*100:5.1f}% {bar}')

    return pred_class, confidence

# Cara pakai:
# predict_road_image('/content/foto_jalan.jpg')
print('✅ Fungsi prediksi siap!')
print('   Cara pakai: predict_road_image("/path/foto_jalan.jpg")')

---
## ✅ Selesai!

### 📁 File Output yang Dihasilkan:
| File | Keterangan |
|------|------------|
| `resnet50_jalan_final.h5` | Model ResNet-50 final |
| `resnet50_jalan_phase1_best.h5` | Model terbaik Phase 1 |
| `resnet50_jalan_phase2_best.h5` | Model terbaik Phase 2 |
| `sample_dataset_jalan.png` | Sample gambar dataset |
| `distribusi_kelas_jalan.png` | Grafik distribusi kelas |
| `resnet50_jalan_history_p1.png` | Grafik training Phase 1 |
| `resnet50_jalan_history_p2.png` | Grafik training Phase 2 |
| `resnet50_jalan_confusion_matrix.png` | Confusion Matrix |
| `resnet50_jalan_per_class_metrics.png` | Grafik Precision/Recall/F1 |
| `resnet50_jalan_classification_report.txt` | Laporan klasifikasi |

### ➡️ Langkah Selanjutnya:
Setelah ResNet-50 selesai → lanjut ke **Baseline CNN** dengan studi kasus yang sama!